# Quantum Error Mitigation Experiment

**Produced by a multi-agent pipeline** (ernest SDK + 4 agents, real OpenRouter runs) — the code cells below are the agent outputs **executed locally with Qiskit 2.x / Aer 0.17**; the stdout cells contain the real run output.

## Experiment design (quantum-theorist)

```
The calculation for the fidelity after applying single-qubit depolarization yields:

\[
F = (1 - 0.02) + \frac{0.02}{3} = 0.98667
\]

This result confirms that the fidelity is approximately 0.98667 after considering the 2% depolarizing noise for single-qubit gates. It further reinforces the conclusion that while error mitigation can improve measurement reliability, residual effects from gate depolarization will prevent the fidelity from reaching 1.0.

### Summary of Experiment Design Specs

1. **Circuit:**
   - **Initial State:** |00⟩
   - **Gates Used:** Hadamard (H) followed by CNOT (CX)

2. **Noise Model:**
   - **Depolarizing Noise:**
     - Single-qubit gates: \( p = 0.02 \)
     - CNOT gates: \( p = 0.04 \)
   - **Readout Error Matrix:**
     \[
     \begin{pmatrix}
     0.95 & 0.05 \\
     0.03 & 0.97
     \end{pmatrix}
     \]

3. **Metrics:**
   - **Hellinger Fidelity:**
   \[
   F_H(P, Q) = \frac{1}{\sqrt{2}} \left( \sum_i \sqrt{p_i \cdot q_i} \right)^2
   \]
   - **Error Rate:**
   \[
   \text{Error Rate} = 1 - F_H
   \]

4. **Error Mitigation Plan:**
   - **Readout Calibration Circuits:** Prepare and measure the states |00⟩, |01⟩, |10⟩, and |11⟩.
   - **Assignment Matrix Construction:** Based on noisy calibration counts.
   - **Count Correction:** Apply least squares for adjusting counts based on the assignment matrix.

5. **Expected Effects:**
   - Fidelity should improve due to readout mitigation but will not reach 1.0 because of residual gate depolarization.
   - **Decision Threshold:** Mitigation is required if fidelity falls below 0.95.

This structured design outlines a thorough approach for executing a quantum noise experiment, facilitating the understanding of quantum systems' behavior in realistic noise conditions while evaluating the effectiveness of proposed error mitigation strategies.
```

In [2]:
# Run 1 — raw noisy execution (quantum-engineer, v1)
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.quantum_info import hellinger_fidelity
import json
import numpy as np

# Initialize the Quantum Circuit
qc = QuantumCircuit(2)
qc.h(0)  # Hadamard gate on qubit 0
qc.cx(0, 1)  # CNOT gate from qubit 0 to qubit 1
qc.measure_all()  # Measure all qubits

# Create the noise model
nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(0.02, 1), ['u1', 'u2', 'u3'])
nm.add_all_qubit_quantum_error(depolarizing_error(0.04, 2), ['cx'])
nm.add_all_qubit_readout_error(ReadoutError([[0.95, 0.05], [0.03, 0.97]]))

# Simulate with noise
sim = AerSimulator(noise_model=nm)
result = sim.run(qc, shots=4096).result()
noisy_counts = result.get_counts()

# Ideal simulation without noise
ideal_sim = AerSimulator()
ideal_result = ideal_sim.run(qc, shots=4096).result()
ideal_counts = ideal_result.get_counts()

# Normalize counts to probability distributions
ideal_probs = {k: v / 4096 for k, v in ideal_counts.items()}
noisy_probs = {k: v / 4096 for k, v in noisy_counts.items()}

# Calculate Hellinger fidelity
fidelity = hellinger_fidelity(ideal_probs, noisy_probs)
error_rate = 1 - fidelity

# Prepare the output dictionary
output = {
    "ideal_counts": ideal_counts,
    "noisy_counts": noisy_counts,
    "fidelity": fidelity,
    "error_rate": error_rate,
}

{"ideal_counts": {"00": 2009, "11": 2087}, "noisy_counts": {"11": 1881, "01": 198, "00": 1824, "10": 193}, "fidelity": 0.9045379882356077, "error_rate": 0.09546201176439228}


## Run 1 results (real)

ideal counts: 00: 2009, 11: 2087

noisy counts: 00: 1824, 01: 198, 10: 193, 11: 1881

fidelity: 0.9045

error_rate: 0.0955

## Analyst decision

```
### Analysis of Execution Report

**Measurement Counts:**
- **Ideal counts:** 
  - \(00: 2009\)
  - \(11: 2087\)
- **Noisy counts:**
  - \(00: 1824\)
  - \(01: 198\)
  - \(10: 193\)
  - \(11: 1881\)

**Fidelity and Error Rate:**
- **Observed fidelity:** 0.9045
- **Error rate:** 0.0955 (derived from noisy counts)

**Total Counts:**
- Total ideal counts = \( 2009 + 2087 = 4096 \)
- Total noisy counts = \( 1824 + 198 + 193 + 1881 = 4096 \)

**Dominant Error Source:**
- The examination of noisy counts reveals that both \(01\) and \(10\) have significant counts, in conjunction with a reduced count for \(00\) and an increased count for \(11\).
- Since the observed negative outcomes (01 and 10) suggest deviations from the expected outcomes, it signifies that the errors are primarily occurring in the **readout** process rather than in gate operations.

### Decision
Since the fidelity (0.9045) is below the threshold of 0.95, **error mitigation is REQUIRED**.

### Change Request for Engineer
To address the errors encountered during the execution, please implement the following changes:

1. **Circuits to Build:**
   - Construct a readout calibration circuit designed to correct the measurement outcomes based on the analysis of fidelity and error rates.

2. **Assignment Matrix:**
   - Compute the assignment correction matrix using the observed noisy counts:
     - This matrix should be derived by analyzing the probabilities associated with each measurement to realign the noisy results with the ideal counts. Focus will be needed on the bias introduced by \(01\) and \(10\) occurrences.

3. **Correction Process:**
   - Implement a readout correction algorithm that applies the computed assignment matrix in subsequent runs, adjusting the readout values to approach the ideal counts more closely.

Please ensure all modifications are in place before proceeding to the next experimental run to duly improve fidelity and reduce error rates.

**VERDICT: MITIGATION REQUIRED**
```

In [4]:
# Run 2 — with readout error mitigation (quantum-engineer, v2)
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.quantum_info import hellinger_fidelity
import json
import numpy as np

# Define the noise model
nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(0.02, 1), ['u1', 'u2', 'u3'])
nm.add_all_qubit_quantum_error(depolarizing_error(0.04, 2), ['cx'])
nm.add_all_qubit_readout_error(ReadoutError([[0.95, 0.05], [0.03, 0.97]]))

# Create the simulator with the noise model
sim = AerSimulator(noise_model=nm)

# Ideal counts and noisy counts
ideal_counts = {'00': 2009, '11': 2087}
noisy_counts = {'00': 1824, '01': 198, '10': 193, '11': 1881}

# Total counts
total_counts = 4096

# Calculate ideal probabilities
ideal_probs = {bs: count / total_counts for bs, count in ideal_counts.items()}
noisy_probs = {bs: count / total_counts for bs, count in noisy_counts.items()}

# Calculate fidelity and error rate
fidelity = hellinger_fidelity(ideal_probs, noisy_probs)
error_rate = 1 - fidelity  # 1 - Fidelity

# Calibration circuits for assignment matrix
basis = ['00', '01', '10', '11']
cal = {}
for bs in basis:
    c = QuantumCircuit(2, 2)
    for i, b in enumerate(bs):
        if b == '1': c.x(i)
    c.measure([0, 1], [0, 1])
    cal[bs] = sim.run(c, shots=4096).result().get_counts()

# Construct assignment matrix
A = np.zeros((4, 4))  # assignment matrix A[observed][true]
for j, bs in enumerate(basis):
    for i, obs in enumerate(basis):
        A[i, j] = cal[bs].get(obs, 0) / total_counts

# Prepare observed noisy counts
n = np.array([noisy_counts.get(bs, 0) for bs in basis], dtype=float)

# Apply least-squares correction to get mitigated counts
x, *_ = np.linalg.lstsq(A, n, rcond=None)
mitigated_counts = {bs: round(max(0.0, xi)) for bs, xi in zip(basis, x)}

# Calculate mitigated probabilities and fidelity
mitigated_probs = {bs: count / total_counts for bs, count in mitigated_counts.items()}
mitigated_fidelity = hellinger_fidelity(ideal_probs, mitigated_probs)
mitigated_error_rate = 1 - mitigated_fidelity  # 1 - Mitigated Fidelity

# Diagnostics
print('Ideal Counts:', ideal_counts)
print('Noisy Counts:', noisy_counts)
print('Fidelity:', fidelity)
print('Error Rate:', error_rate)
print('Mitigated Counts:', mitigated_counts)
print('Mitigated Fidelity:', mitigated_fidelity)
print('Mitigated Error Rate:', mitigated_error_rate)

# Final output
output = {
    'ideal_counts': ideal_counts,
    'noisy_counts': noisy_counts,
    'fidelity': fidelity,
    'error_rate': error_rate,
    'mitigated_counts': mitigated_counts,
    'mitigated_fidelity': mitigated_fidelity,
    'mitigated_error_rate': mitigated_error_rate
}

Ideal Counts: {'00': 2009, '11': 2087}
Noisy Counts: {'00': 1824, '01': 198, '10': 193, '11': 1881}
Fidelity: 0.9045379882356077
Error Rate: 0.09546201176439228
Mitigated Counts: {'00': 2025, '01': 34, '10': 44, '11': 1993}
Mitigated Fidelity: 0.9807781519063344
Mitigated Error Rate: 0.01922184809366556
{"ideal_counts": {"00": 2009, "11": 2087}, "noisy_counts": {"00": 1824, "01": 198, "10": 193, "11": 1881}, "fidelity": 0.9045379882356077, "error_rate": 0.09546201176439228, "mitigated_counts": {"00": 2025, "01": 34, "10": 44, "11": 1993}, "mitigated_fidelity": 0.9807781519063344, "mitigated_error_rate": 0.01922184809366556}


## Run 2 results (real)

ideal counts: 00: 2009, 11: 2087

noisy counts: 00: 1824, 01: 198, 10: 193, 11: 1881

fidelity: 0.9045

error_rate: 0.0955

mitigated counts: 00: 2025, 01: 34, 10: 44, 11: 1993

mitigated_fidelity: 0.9808

mitigated_error_rate: 0.0192

## Final verdict (quantum-analyst)

```
### Comparison of RUN 1 and RUN 2

**RUN 1 (raw):**
- **Fidelity:** 0.9045
- **Error Rate:** 0.0955

**RUN 2 (mitigated):**
- **Fidelity:** 0.9808
- **Error Rate:** 0.0192

### Delta Analysis
- **Fidelity Delta:** 
  \[ \text{Delta} = 0.9808 - 0.9045 = 0.0763 \]
- **Error Rate Delta:** 
  \[ \text{Delta} = 0.0955 - 0.0192 = 0.0763 \]

### Judgement on Mitigation Effectiveness
The fidelity increased significantly from 0.9045 in RUN 1 to 0.9808 in RUN 2, reflecting a notable improvement of 0.0763. Simultaneously, the error rate decreased from 0.0955 to 0.0192, indicating effective mitigation of readout errors.

### Remaining Error Explanation
Despite the successful mitigation of readout errors, the residual error remains due to **gate depolarization** effects, which are not addressed by readout calibration techniques. This indicates that while readout errors have been substantially corrected, further sources of error, particularly those resulting from the gate operations, still impact the overall fidelity and error rates.

### Final Verdict
Given the significant improvement in fidelity and reduction in error rates, the mitigation appears effective, but the remaining gate-related errors indicate that further improvements may be necessary.

**VERDICT: FURTHER MITIGATION NEEDED**
```